<a href="https://colab.research.google.com/github/Vini364/chlorine-study/blob/main/CNNWithoutCropping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
import json
import base64
import io
import numpy as np
import torch
from PIL import Image
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from pillow_heif import register_heif_opener
from torchvision.io import decode_image, ImageReadMode, decode_heic


In [10]:

with open('/content/drive/MyDrive/Research/new_dataset.json') as f:
  dataset = json.load(f)
records = []
for index, item in enumerate(dataset):
  records.append({
      "image": item["image"],
      "concentration": float(item["concentration"]),
      "source_id": item.get("source_id", f"sample{index}.jpg"),
  })

In [54]:

#This method essentially decodes our image from JSON to PIL Object
def decode_images(image_data): #Decodes the image
  if isinstance(image_data, str):
    image_bytes = base64.b64decode(image_data)    #Converts the json data into raw bytes
    return Image.open(io.BytesIO(image_bytes)).convert("RGB")   #wraps the raw bytes into a file-like object for the PIL library to convert to rgb values.
  raise ValueError("Unsupported image format")

#Takes the dataset, and wraps into a tensor form, in such a way that the DataLoader can work with it
class ConcentrationDataset(Dataset): #Essentially creating a torch dataset
  def __init__(self, records, transform = None):
    self.records = records #Dictionary of the json dataset
    self.transform = transform
    self.labels = [r["concentration"] for r in records]
    self.images = [decode_images(r["image"]) for r in records]
  def __len__(self):
    return len(self.records)
  def __getitem__(self, idx): #helps it retrieve a specific image based on the index
    image = self.images[idx]
    if self.transform:
      image = self.transform(image)
    label = torch.tensor(self.labels[idx], dtype = torch.float32)
    return image, label

#A way to create a transformer, including for the training dataset such that it augments the data.
def get_transforms():
  train_transform = transforms.Compose([    #transformer for the training dataset, including augmentation such as flipping, color augmentation.
      transforms.RandomVerticalFlip(),
      transforms.RandomHorizontalFlip(),
      transforms.Resize((224, 224)),
      transforms.ColorJitter(0.2, 0.2, 0.2),
      transforms.ToTensor(),                #Converting from PIL object to tensor object
      transforms.Normalize(mean = [0.485, 0.456, 0.406],
                           std = [0.229, 0.224, 0.225]), #Normalizing for Image Net specifications
    ])
  val_transform = transforms.Compose([     #Transformer for the validation dataset, not performing augmentation to ensure consistent evaluation.
      transforms.Resize((224,224)),
      transforms.ToTensor(),
      transforms.Normalize(mean = [0.485, 0.456, 0.406],
                          std = [0.229, 0.224, 0.225]),
  ])
  return train_transform, val_transform

#Building the model itself, find the text attached below for more info.

def build_model(backbone = "efficientnet_b0", freeze_backbone = True):
  if backbone == "efficientnet_b0":
    model = models.efficientnet_b0(weights = "IMAGENET1K_V1")
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p = 0.3),
         nn.Linear(in_features, 1))
    if freeze_backbone:
      for name, param in model.named_parameters():
        if "classifier" not in name:
          param.requires_grad = False
  elif backbone == "resnet18":
    model = models.resnet18(weights ="IMAGENET1K_V1")
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, 1)
    if freeze_backbone:
      for name, param in model.named_parameters():
        if "fc" not in name:
          param.requires_grad = False
  return model.to(DEVICE)
def evaluate(model, loader):
  model.eval()
  all_preds, all_labels = [], []
  with torch.no_grad(): #Making sure that the models weights arent updated
    for images, labels in loader:
      images, labels = images.to(DEVICE), labels.to(DEVICE)
      preds = model(images).squeeze() #so that only the tensor of predictions are given
      all_preds.extend(preds.cpu().numpy()) #converting to numpy since only cpu can do those operations.
      all_labels.extend(labels.cpu().numpy())
  all_preds = np.array(all_preds)
  all_labels = np.array(all_labels)
  return{
      "r2" : float(r2_score(all_labels, all_preds)),
      "rmse": float(mean_squared_error(all_labels, all_preds)**0.5),
      "mae" : float(mean_absolute_error(all_labels, all_preds)),
  }


So essentially for the training portion, especially with transfer learning, the models we've got from the internet already have their predefined weights. We dont want to mess with those weights too much otherwise we've just made the model overfit for us and potentially forget everything else it was taught. Thus for the first training we freeze all the layers except the last outer layer, which've transformed from the 1000 output, to our 1 output or the regression output. Thus, we fiddle with the weights during this training. After a certain number of epochs, we unfreeze the layer, but use a very small learning rate to not rattle or bias the weights too much. Then we see how our model holds against our validation set, hopefully it performs well. We can ofcourse toy around with the number of epochs frozen/unfrozen, and batch size. For tha latter, considering our dataset isnt that large, I dont suppose we're going to see too much of a difference, and too small a number will only serve to make the process longer and add irrelevant noise, while too large will make it fail to generalise well.

In [14]:
def train_cnn(records, backbone = "efficientnet_b0", epochs_frozen = 10, epochs_unfrozen = 20, batch_size = 16, lr_frozen = 1e-3, lr_unfrozen = 1e-4):
  source_ids = list(set(r["source_id"] for r in records))
  train_ids, validation_ids = train_test_split(source_ids, test_size = 0.2, random_state = 42)
  train_ids = set(train_ids) #Using sets since it introduces faster hashing, O(1) lookup later for the train and validation records
  validation_ids = set(validation_ids)
  train_records = [r for r in records if r["source_id"] in train_ids]
  validation_records = [r for r in records if r["source_id"] in validation_ids]
  train_transform, validation_transform = get_transforms()

  train_dataset = ConcentrationDataset(train_records, transform = train_transform)
  validation_dataset = ConcentrationDataset(validation_records, transform = validation_transform)
  train_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True)
  validation_loader = DataLoader(validation_dataset, batch_size = batch_size)

  model = build_model(backbone = backbone, freeze_backbone=True)
  optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr = lr_frozen)
  criteria = nn.MSELoss()
  print("Frozen Layer:")
  for epoch in range(epochs_frozen):
    print(f"Starting epoch number {epoch}")
    model.train()
    total_loss = 0
    for image, label in train_loader:
      image, label = image.to(DEVICE), label.to(DEVICE)
      optimizer.zero_grad() #Sets the weights to 0 for forward pass
      preds = model(image).squeeze()
      loss = criteria(preds, label)
      loss.backward() #calculates loss gradients for each parameter
      optimizer.step() #updates the parameters depending on the loss function
      total_loss+= loss.item() #Adds to the total loss
    if (epoch + 1) % 5 == 0:
      metrics = evaluate(model, validation_loader)
      print(f"Epoch {epoch + 1},  loss = {total_loss/len(train_loader):.4f}")
      print(f"R^2 = {metrics['r2']:.4f}, RMSE = {metrics['rmse']:.4f}")

  print("\n Fine Tuning with unfrozen layer")
  for param in model.parameters():
    param.requires_grad = True
  optimizer = torch.optim.Adam(model.parameters(), lr = lr_unfrozen)
  scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max = epochs_unfrozen)
  for epoch in range(epochs_unfrozen):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
      images, labels = images.to(DEVICE), labels.to(DEVICE)
      optimizer.zero_grad()
      preds = model(images).squeeze()
      loss = criteria(preds, labels)
      loss.backward()
      optimizer.step()
      total_loss += loss.item()
    scheduler.step()
    if (epoch+1) % 5 ==0:
      metrics = evaluate(model, validation_loader)
      print(f"Epoch {epoch+1}, loss = {total_loss/len(train_loader)}")
      print(f"R^2 = {metrics['r2']:.4f}, RMSE = {metrics['rmse']:.4f}")
  train_evaluation = evaluate(model, train_loader)
  validation_evaluation = evaluate(model, validation_loader)
  r2_gap = train_evaluation['r2'] - validation_evaluation['r2']

  print(f"Final Results for {backbone}")
  print(f"Training R^2 = {train_evaluation['r2']:.4f}")
  print(f"Test R^2: {validation_evaluation['r2']:.4f}")
  print(f"Test RMSE: {validation_evaluation['rmse']:.4f}")
  print(f"Test MAE: {validation_evaluation['mae']:.4f}")
  print(f"R^2 GAP: {r2_gap:.4f}")
  torch.save(model.state_dict(), f"/content/drive/MyDrive/Research/cnn_{backbone}.pth")
  return model, train_evaluation, validation_evaluation



In [15]:
for backbone in ["efficientnet_b0", "resnet18"]:
  train_cnn(records, backbone = backbone)

Frozen Layer:
Starting epoch number 0
Starting epoch number 1
Starting epoch number 2
Starting epoch number 3
Starting epoch number 4
Epoch 5,  loss = 0.1578
R^2 = 0.6249, RMSE = 0.3645
Starting epoch number 5
Starting epoch number 6
Starting epoch number 7
Starting epoch number 8
Starting epoch number 9
Epoch 10,  loss = 0.1263
R^2 = 0.7892, RMSE = 0.2733

 Fine Tuning with unfrozen layer
Epoch 5, loss = 0.0721148071081742
R^2 = 0.8836, RMSE = 0.2031
Epoch 10, loss = 0.05865031820924386
R^2 = 0.9300, RMSE = 0.1575
Epoch 15, loss = 0.049379889650837235
R^2 = 0.9477, RMSE = 0.1362
Epoch 20, loss = 0.04985880126933689
R^2 = 0.9548, RMSE = 0.1265
Final Results for efficientnet_b0
Training R^2 = 0.9567
Test R^2: 0.9548
Test RMSE: 0.1265
Test MAE: 0.1069
R^2 GAP: 0.0019
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 196MB/s]


Frozen Layer:
Starting epoch number 0
Starting epoch number 1
Starting epoch number 2
Starting epoch number 3
Starting epoch number 4
Epoch 5,  loss = 0.1183
R^2 = 0.6017, RMSE = 0.3756
Starting epoch number 5
Starting epoch number 6
Starting epoch number 7
Starting epoch number 8
Starting epoch number 9
Epoch 10,  loss = 0.1080
R^2 = 0.6556, RMSE = 0.3493

 Fine Tuning with unfrozen layer
Epoch 5, loss = 0.055140366413347096
R^2 = 0.9343, RMSE = 0.1526
Epoch 10, loss = 0.04269428713166196
R^2 = 0.9696, RMSE = 0.1038
Epoch 15, loss = 0.015986426024819197
R^2 = 0.9717, RMSE = 0.1001
Epoch 20, loss = 0.02016326746619914
R^2 = 0.9783, RMSE = 0.0876
Final Results for resnet18
Training R^2 = 0.9828
Test R^2: 0.9783
Test RMSE: 0.0876
Test MAE: 0.0686
R^2 GAP: 0.0045


In [10]:
model = build_model(backbone = "resnet18", freeze_backbone = False)
model.load_state_dict(torch.load('/content/drive/MyDrive/Research/cnn_resnet18.pth'))

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [71]:
register_heif_opener()
train_transformer, val_transformer = get_transforms()
def predict_single_image(image_file_path):
  img = Image.open(image_file_path).convert("RGB")
  img_tensor = val_transformer(img).unsqueeze(0).to(DEVICE)
  with torch.no_grad():
    model.eval()
    pred = model(img_tensor)
  print(pred.item())

In [72]:
predict_single_image('/content/drive/MyDrive/Aresty_Data_2/0.36/IMG_9077.HEIC')

0.255524605512619
